In [ ]:
# Cell 1 — 공통 실행 환경 설정
# 이 Cell은 모든 tutorial Notebook의 첫 번째 Cell이며 여러 번 실행해도 안전합니다.

from pathlib import Path
import importlib.util
import os
import subprocess
import sys

# Google Drive의 project folder가 다르면 이 값만 수정합니다.
COLAB_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/DiffDodree"
)

# 필요하면 환경변수로 project root를 직접 지정할 수 있습니다.
PROJECT_ROOT_OVERRIDE = os.environ.get("GUGAKGEN_PROJECT_ROOT")


def find_local_project_root(start: Path) -> Path:
    """현재 위치에서 pyproject.toml을 기준으로 project root를 찾습니다."""
    candidates = [start.resolve(), *start.resolve().parents]

    for candidate in candidates:
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "project root를 찾지 못했습니다. "
        "GUGAKGEN_PROJECT_ROOT를 지정하거나 project folder에서 Notebook을 실행하세요."
    )


try:
    # 순수 local Python에는 상위 ``google`` namespace가 없을 수 있습니다.
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    # 이미 mount되어 있으면 기존 mount를 그대로 사용합니다.
    drive.mount("/content/drive", force_remount=False)

if PROJECT_ROOT_OVERRIDE:
    project_root = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
elif IN_COLAB:
    project_root = COLAB_PROJECT_ROOT.resolve()
else:
    project_root = find_local_project_root(Path.cwd())

if not project_root.exists():
    raise FileNotFoundError(
        f"project folder가 존재하지 않습니다: {project_root}\n"
        "COLAB_PROJECT_ROOT 또는 GUGAKGEN_PROJECT_ROOT를 확인하세요."
    )

bootstrap_script = project_root / "scripts" / "bootstrap_notebook.py"

if not bootstrap_script.exists():
    raise FileNotFoundError(
        f"Notebook bootstrap script를 찾지 못했습니다: {bootstrap_script}"
    )

environment_profile = "colab" if IN_COLAB else "local"

# 필요한 dependency만 확인하고 누락된 package만 설치합니다.
# Colab profile에서는 Colab이 제공한 CUDA용 PyTorch를 교체하지 않습니다.
subprocess.run(
    [
        sys.executable,
        str(bootstrap_script),
        "--project-root",
        str(project_root),
        "--profile",
        environment_profile,
        "--install-missing",
    ],
    check=True,
)

# 현재 kernel에서 src package를 즉시 import할 수 있게 합니다.
src_root = project_root / "src"

if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

os.environ["GUGAKGEN_PROJECT_ROOT"] = str(project_root)
os.chdir(project_root)

from gugakgen.device import resolve_device
from gugakgen.reproducibility import set_global_seed

runtime = resolve_device(preference="auto")
device = runtime.device

SEED = 2026
set_global_seed(SEED)

print("환경 설정 완료")
print(f"- 실행 환경: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"- Project root: {project_root}")
print(f"- Device: {device}")
print(f"- AMP enabled: {runtime.amp_enabled}")
print(f"- Seed: {SEED}")

# 01. 양청도드리 Data inspection

이 Notebook은 전문가가 전사하고 마디 단위로 구분한 양청도드리 피리 악보를 불러와
Phase 1의 symbolic representation을 확인합니다. MusicXML의 pitch와 duration을 그대로
읽으며, 율명 대응은 전문가가 검증한 mapping만 사용합니다.

이 단계에서는 시김새, 농현, 음색, 미세 timing을 추론하지 않습니다. 확인 대상은 악보의
마디, pitch token, duration token, family label입니다.

In [ ]:
# Piece configuration과 MusicXML source를 확인합니다.
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import display

from gugakgen.config import load_yaml
from gugakgen.data.musicxml import combine_scores, parse_musicxml, score_to_tokens
from gugakgen.data.vocabulary import ScoreVocabulary
from gugakgen.structure.transition_graph import TransitionGraph

piece_config = load_yaml(project_root / "configs/yangcheongdodeuri.yaml")
piece_info = piece_config["piece"]
data_config = piece_config["data"]
chapter_ranges = tuple(tuple(item["measures"]) for item in piece_config["chapters"])

source_paths = []
for pattern in data_config["source_globs"]:
    source_paths.extend(sorted(project_root.glob(pattern)))

if len(source_paths) != 2:
    raise RuntimeError(f"피리 MusicXML 2개를 예상했지만 {len(source_paths)}개를 찾았습니다: {source_paths}")

display(pd.DataFrame({"MusicXML source": [str(path.relative_to(project_root)) for path in source_paths]}))

In [ ]:
# 두 MusicXML segment를 72개 마디의 단일 악보로 연결합니다.
segments = []
measure_offset = 0
default_meters = ((4, 4), (6, 4))

for source_path, default_meter in zip(source_paths, default_meters):
    segment = parse_musicxml(
        source_path,
        piece_id=piece_info["id"],
        part_name=piece_info["part"],
        measure_offset=measure_offset,
        default_meter=default_meter,
        chapter_ranges=chapter_ranges,
        reject_chords=data_config["reject_chords"],
    )
    segments.append(segment)
    measure_offset += len(segment.measures)

score = combine_scores(segments)
score.validate_meters()

vocabulary = ScoreVocabulary.from_json(project_root / data_config["vocabulary"])
transition_graph = TransitionGraph.from_json(project_root / data_config["transition_graph"])
tokens = score_to_tokens(
    score,
    vocabulary,
    original_path=transition_graph.original_path,
    num_position_buckets=4,
)

summary = pd.DataFrame(
    [
        {"항목": "악곡", "값": piece_info["title_ko"]},
        {"항목": "파트", "값": piece_info["part"]},
        {"항목": "마디 수", "값": len(score.measures)},
        {"항목": "event 수", "값": sum(len(measure.events) for measure in score.measures)},
        {"항목": "pitch token 수", "값": vocabulary.num_pitch_tokens},
        {"항목": "duration token 수", "값": vocabulary.num_duration_tokens},
        {"항목": "family 수", "값": vocabulary.num_families},
        {"항목": "최대 event/마디", "값": vocabulary.max_events_per_measure},
    ]
)
display(summary)

In [ ]:
# 전문가가 정한 마디별 symbolic record와 token을 함께 확인합니다.
measure_rows = []
for index, measure in enumerate(score.measures):
    measure_rows.append(
        {
            "마디": measure.number,
            "장": measure.chapter,
            "장단": f"{measure.meter.numerator}/{measure.meter.denominator}",
            "event 수": len(measure.events),
            "pitch": [event.pitch for event in measure.events],
            "duration_q": [float(event.duration_q) for event in measure.events],
            "family": transition_graph.original_path[index],
        }
    )

measure_table = pd.DataFrame(measure_rows)
display(measure_table.head(12))
print("Token array shape:", tokens.pitch.shape)
print("첫 마디 pitch token:", tokens.pitch[0].tolist())
print("첫 마디 duration token:", tokens.duration[0].tolist())

In [ ]:
# 전문가 검증 율명 mapping으로 pitch distribution을 표시합니다.
yulmyung = {int(midi): name for midi, name in piece_config["pitch"]["midi_to_yulmyung"].items()}
pitch_counts = Counter(
    event.pitch
    for measure in score.measures
    for event in measure.events
    if event.pitch is not None
)
pitch_table = pd.DataFrame(
    [
        {"MIDI pitch": pitch, "율명": yulmyung.get(pitch, str(pitch)), "event 수": count}
        for pitch, count in sorted(pitch_counts.items())
    ]
)
display(pitch_table)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(pitch_table["율명"], pitch_table["event 수"], color="#4c6ef5")
ax.set_title("양청도드리 피리 pitch distribution")
ax.set_xlabel("율명")
ax.set_ylabel("event 수")
ax.grid(axis="y", alpha=0.2)
plt.show()

In [ ]:
# 원하는 마디를 선택하여 record와 token을 대화식으로 확인합니다.
import ipywidgets as widgets


def inspect_measure(measure_number: int = 1) -> None:
    index = measure_number - 1
    display(measure_table.iloc[[index]])
    print("pitch token   :", tokens.pitch[index].tolist())
    print("duration token:", tokens.duration[index].tolist())
    print("family token  :", int(tokens.family[index]))


widgets.interact(
    inspect_measure,
    measure_number=widgets.IntSlider(value=1, min=1, max=len(score.measures), step=1, description="마디"),
)

## 확인 기준

- 모든 마디의 duration 합이 해당 장단과 정확히 일치해야 합니다.
- pitch 집합은 전문가가 검증한 율명 mapping의 범위 안에 있어야 합니다.
- PAD와 MASK는 음악 event가 아니라 Transformer 입력을 위한 특별 token입니다.
- 이 Notebook의 결과는 Data validation이며 생성 성능에 대한 결론이 아닙니다.